# Exo: IK → ID vs OpenSim ID (exo torque removed)

Compares checkpoint inference from telemetry IK inputs against OpenSim inverse dynamics for **hip-exo** and **knee-exo** trials.

- **OpenSim data**: `/media/metamobility3/Samsung_T52/Results/processed`
- **Telemetry**: `os_kinetics/*_exo_on.npz`
- **Sync**: GPIO sync pulse **rising edge** (telemetry vs mocap)
- **GT**: OpenSim ID minus applied exo torque (`applied_R`, etc.) → N·m/kg
- **Filters**: match hip/knee training configs (causal 6 Hz input, zero-phase 6 Hz output)
- **Metrics**: RMSE and R² only

Use kernel `jinwoo-addbiomech` (needs PyTorch).

In [1]:
import csv
import io
import json
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.signal import butter, sosfilt, sosfiltfilt

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
TELEMETRY_ROOT = PROJECT_ROOT
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SUBJECT_TOKEN_TO_DIR = {
    'ab01_jinwoo': 'AB01_Jinwoo', 'ab02_oscar': 'AB02_Oscar', 'ab03_ilseung': 'AB03_Ilseung',
    'ab04_changseob': 'AB04_Changseob', 'ab05_maria': 'AB05_Maria', 'ab06_jimin': 'AB06_Jimin',
    'ab07_amy': 'AB07_Amy', 'ab08_seokhyun': 'AB08_Seokhyun',
}
SUBJECT_MASS_KG = {
    'ab01_jinwoo': 88.0, 'ab02_oscar': 71.1, 'ab03_ilseung': 84.4, 'ab04_changseob': 74.0,
    'ab05_maria': 55.0, 'ab06_jimin': 82.6, 'ab07_amy': 51.3, 'ab08_seokhyun': 71.9,
}

HIP_CHECKPOINT = PROJECT_ROOT / 'runs/0512_ik_id_hip_causal_in_zero_out/best_model.pt'
KNEE_CHECKPOINT = PROJECT_ROOT / 'runs/0512_ik_id_knee_causal_in_zero_out/best_model.pt'
GT_LPF_CUTOFF, GT_LPF_ORDER, GT_LPF_MODE = 6.0, 4, 'zero_phase'
HIP_OUTPUT_DELAY_S = 0.0
KNEE_OUTPUT_DELAY_S = 0.0

sys.path.insert(0, str(PROJECT_ROOT))
from model import TCN

print(f'Processed root: {PROCESSED_ROOT}')
print(f'Telemetry root: {TELEMETRY_ROOT}')
print(f'Device: {DEVICE}')

Processed root: /media/metamobility3/Samsung_T52/Results/processed
Telemetry root: /home/metamobility3/Jinwoo/os_kinetics
Device: cuda


In [2]:
def infer_fs_hz(time_s, default_fs=100.0):
    if time_s is None or len(time_s) < 3:
        return float(default_fs)
    dt = np.diff(np.asarray(time_s, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return float(1.0 / np.median(dt)) if dt.size else float(default_fs)


def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * float(fs_hz)
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(arr) < 4:
        return arr.astype(np.float32)
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    y = sosfiltfilt(sos, arr) if mode == 'zero_phase' else sosfilt(sos, arr)
    return y.astype(np.float32)


def lpf_nan(x, fs_hz, cutoff_hz, order, mode):
    arr = np.asarray(x, dtype=np.float32).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(3, order + 1):
        return arr
    if finite.all():
        return butter_lpf(arr, fs_hz, cutoff_hz, order, mode)
    idx = np.arange(arr.size, dtype=np.float64)
    filled = np.interp(idx, idx[finite], arr[finite]).astype(np.float32)
    out = butter_lpf(filled, fs_hz, cutoff_hz, order, mode)
    out[~finite] = np.nan
    return out


def delay_series(x, fs_hz, delay_s):
    y = np.asarray(x, dtype=np.float32).copy()
    n = int(round(float(delay_s) * float(fs_hz)))
    if n <= 0:
        return y
    out = np.full_like(y, np.nan)
    if n < y.size:
        out[n:] = y[:-n]
    return out


def rmse_r2(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    r2 = float(1.0 - ss_res / (ss_tot + 1e-12))
    return rmse, r2


def _subject_token(stem: str) -> str:
    return '_'.join(stem.lower().split('_')[:2])


def subject_dir_from_stem(stem: str) -> Path:
    token = _subject_token(stem)
    name = SUBJECT_TOKEN_TO_DIR.get(token)
    if name is None:
        raise FileNotFoundError(f'Unknown subject token: {token}')
    p = PROCESSED_ROOT / name
    if not p.is_dir():
        raise FileNotFoundError(f'Processed subject folder missing: {p}')
    return p


def trial_cond_speed(stem: str) -> Tuple[str, str]:
    parts = stem.lower().split('_')
    return parts[4].upper(), parts[3]


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def parse_mocap_gpio(path: Path):
    with open(path) as f:
        _ = f.readline()
        try:
            fs_hz = float(f.readline().strip().split(',')[0])
        except Exception:
            fs_hz = 1000.0
        _ = f.readline()
        header = f.readline().strip().split(',')
        _ = f.readline()
        trig_idx = next((header.index(c) for c in ('trig - Electric Potential', 'jet', 'trig', 'gpio') if c in header), None)
        if trig_idx is None:
            raise KeyError(f'No trigger column in {path.name}')
        vals = []
        for row in csv.reader(f):
            if len(row) > trig_idx:
                try:
                    vals.append(float(row[trig_idx]))
                except ValueError:
                    pass
    gpio = np.asarray(vals, dtype=np.float64)
    return np.arange(gpio.size) / fs_hz, gpio


def first_rising_edge(x, threshold=0.5):
    xx = np.asarray(x, dtype=np.float64)
    good = np.isfinite(xx[:-1]) & np.isfinite(xx[1:])
    idx = np.where(good & (xx[:-1] <= threshold) & (xx[1:] > threshold))[0]
    return int(idx[0] + 1) if idx.size else None


def gpio_offset_s(t_exo, g_exo, t_mocap, g_mocap):
    idx_exo = first_rising_edge(g_exo)
    idx_mocap = first_rising_edge(g_mocap)
    if idx_exo is None or idx_mocap is None:
        return None, idx_exo, idx_mocap
    return float(t_mocap[idx_mocap] - t_exo[idx_exo]), idx_exo, idx_mocap


def extract_gpio(npz):
    for k in ('gpio_output', 'GPIO', 'gpio', 'trigger'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float32), k
    raise KeyError('No GPIO key in npz')


def extract_applied_r(npz):
    for k in ('applied_torque_R', 'applied_R', 'cmd_R', 'mtr_cmd_R'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float32), k
    raise KeyError('No applied_R torque key in npz')

print('Helpers ready.')

Helpers ready.


In [3]:
def load_checkpoint(path: Path):
    import json
    ckpt = torch.load(str(path), map_location=DEVICE, weights_only=False)
    with open(path.parent / 'config.json') as f:
        train_cfg = json.load(f)
    cfg = dict(ckpt['model_config'])
    allowed = {k for k in cfg if k in ('n_input_channels', 'n_output_channels', 'hidden_channels', 'n_blocks', 'kernel_size', 'dropout')}
    model = TCN(**{k: cfg[k] for k in allowed})
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    filt = {
        'window': int(ckpt.get('window_size', 100)),
        'in_cutoff': float(train_cfg.get('lowpass_cutoff_hz', 6.0)),
        'in_order': int(train_cfg.get('lowpass_order', 4)),
        'in_mode': str(train_cfg.get('input_lowpass_mode', 'causal')),
        'vel_cutoff': float(train_cfg.get('velocity_lowpass_cutoff_hz', 15.0)),
        'out_cutoff': float(train_cfg.get('lowpass_cutoff_hz', 6.0)),
        'out_order': int(train_cfg.get('lowpass_order', 4)),
        'out_mode': str(train_cfg.get('output_lowpass_mode', 'zero_phase')),
    }
    return model, filt


@torch.no_grad()
def run_tcn_last_step(model, angle, vel, window_size):
    n = min(len(angle), len(vel))
    pred = np.zeros(n, dtype=np.float32)
    for t in range(n):
        start = max(0, t - window_size + 1)
        valid = t - start + 1
        x = np.zeros((2, window_size), dtype=np.float32)
        x[0, -valid:] = angle[start:t + 1]
        x[1, -valid:] = vel[start:t + 1]
        xt = torch.from_numpy(x).unsqueeze(0).to(DEVICE)
        pred[t] = float(model(xt)[0, 0, -1].item())
    return pred


HIP_MODEL, HIP_FILT = load_checkpoint(HIP_CHECKPOINT)
KNEE_MODEL, KNEE_FILT = load_checkpoint(KNEE_CHECKPOINT)
print('Hip filters:', HIP_FILT)
print('Knee filters:', KNEE_FILT)

Hip filters: {'window': 100, 'in_cutoff': 6.0, 'in_order': 4, 'in_mode': 'causal', 'vel_cutoff': 15.0, 'out_cutoff': 6.0, 'out_order': 4, 'out_mode': 'zero_phase'}
Knee filters: {'window': 100, 'in_cutoff': 6.0, 'in_order': 4, 'in_mode': 'causal', 'vel_cutoff': 15.0, 'out_cutoff': 6.0, 'out_order': 4, 'out_mode': 'zero_phase'}


In [4]:
EXO_SPECS = {
    'hip-exo': {
        'pattern': '*_hip_*_exo_on.npz',
        'angle_key': 'model_in_angle_raw',
        'vel_key': 'model_in_vel_raw',
        'moment_col': 'hip_flexion_r_moment',
        'joint': 'hip_flexion_r',
        'model': HIP_MODEL,
        'filt': HIP_FILT,
        'delay_s': HIP_OUTPUT_DELAY_S,
    },
    'knee-exo': {
        'pattern': '*_knee_*_exo_on.npz',
        'angle_key': 'model_in_knee_angle_raw',
        'vel_key': 'model_in_knee_vel_raw',
        'moment_col': 'knee_angle_r_moment',
        'joint': 'knee_angle_r',
        'model': KNEE_MODEL,
        'filt': KNEE_FILT,
        'delay_s': KNEE_OUTPUT_DELAY_S,
    },
}

WARNINGS = []
CANDIDATES = {k: sorted(TELEMETRY_ROOT.glob(v['pattern'])) for k, v in EXO_SPECS.items()}
for exo_kind, files in CANDIDATES.items():
    print(f'{exo_kind}: {len(files)} telemetry files found')
    for p in files:
        print(f'  - {p.name}')

hip-exo: 36 telemetry files found
  - _ab01_jinwoo_hip_0p8mps_lg_exo_on.npz
  - _ab01_jinwoo_hip_0p8mps_ra_exo_on.npz
  - _ab01_jinwoo_hip_1p2mps_lg_exo_on.npz
  - _ab01_jinwoo_hip_1p6mps_lg_exo_on.npz
  - _ab02_oscar_hip_0p8mps_lg_exo_on.npz
  - _ab02_oscar_hip_0p8mps_ra_exo_on.npz
  - _ab02_oscar_hip_1p2mps_lg_exo_on.npz
  - _ab02_oscar_hip_1p6mps_lg_exo_on.npz
  - ab01_jinwoo_hip_0p8mps_lg_exo_on.npz
  - ab01_jinwoo_hip_0p8mps_ra_exo_on.npz
  - ab01_jinwoo_hip_1p2mps_lg_exo_on.npz
  - ab01_jinwoo_hip_1p6mps_lg_exo_on.npz
  - ab02_oscar_hip_0p8mps_lg_exo_on.npz
  - ab02_oscar_hip_0p8mps_ra_exo_on.npz
  - ab02_oscar_hip_1p2mps_lg_exo_on.npz
  - ab02_oscar_hip_1p6mps_lg_exo_on.npz
  - ab03_ilseung_hip_0p8mps_lg_exo_on.npz
  - ab03_ilseung_hip_0p8mps_ra_exo_on.npz
  - ab03_ilseung_hip_1p2mps_lg_exo_on.npz
  - ab03_ilseung_hip_1p6mps_lg_exo_on.npz
  - ab04_changseob_hip_0p8mps_lg_exo_on.npz
  - ab04_changseob_hip_0p8mps_ra_exo_on.npz
  - ab04_changseob_hip_1p2mps_lg_exo_on.npz
  - ab04_c

In [5]:
def evaluate_exo_trial(npz_path: Path, exo_kind: str) -> Optional[Dict]:
    spec = EXO_SPECS[exo_kind]
    stem = npz_path.stem
    cond, speed = trial_cond_speed(stem)
    subj_dir = subject_dir_from_stem(stem)
    mass = SUBJECT_MASS_KG[_subject_token(stem)]

    id_path = subj_dir / exo_kind / 'id' / f'{cond}_{speed}_id.sto'
    mocap_path = subj_dir / exo_kind / 'mocap' / f'{cond}_{speed}.csv'
    if not id_path.exists() or not mocap_path.exists():
        missing = [p.name for p in (id_path, mocap_path) if not p.exists()]
        WARNINGS.append(f'[WARN] {stem}: missing processed files {missing} — skipped')
        return None

    d = np.load(str(npz_path), allow_pickle=True)
    if spec['angle_key'] not in d.files or spec['vel_key'] not in d.files:
        WARNINGS.append(f'[WARN] {stem}: missing model input keys — skipped')
        return None

    t_raw = np.asarray(d['time'], dtype=np.float32) if 'time' in d.files else np.arange(len(d[spec['angle_key']]), dtype=np.float32)
    angle_raw = np.asarray(d[spec['angle_key']], dtype=np.float32)
    vel_raw = np.asarray(d[spec['vel_key']], dtype=np.float32)
    gpio, gpio_key = extract_gpio(d)
    applied_nm, applied_key = extract_applied_r(d)

    n = min(len(t_raw), len(angle_raw), len(vel_raw), len(gpio), len(applied_nm))
    t_raw, angle_raw, vel_raw, gpio, applied_nm = [a[:n] for a in (t_raw, angle_raw, vel_raw, gpio, applied_nm)]

    t_mocap, gpio_mocap = parse_mocap_gpio(mocap_path)
    offset_s, idx_exo, idx_mocap = gpio_offset_s(t_raw, gpio, t_mocap, gpio_mocap)
    if offset_s is None:
        WARNINGS.append(f'[WARN] {stem}: no GPIO rising edge — skipped')
        return None

    cols, id_data = read_sto(id_path)
    t_id = id_data[:, cols.index('time')]
    id_moment_nm = id_data[:, cols.index(spec['moment_col'])]

    t_aligned = t_raw.astype(np.float64) + offset_s
    fs_hz = infer_fs_hz(t_raw)
    fcfg = spec['filt']

    angle_in = butter_lpf(angle_raw, fs_hz, fcfg['in_cutoff'], fcfg['in_order'], fcfg['in_mode'])
    vel_in = butter_lpf(vel_raw, fs_hz, fcfg['vel_cutoff'], fcfg['in_order'], fcfg['in_mode'])
    pred_raw = run_tcn_last_step(spec['model'], angle_in, vel_in, fcfg['window'])
    pred_out = butter_lpf(pred_raw, fs_hz, fcfg['out_cutoff'], fcfg['out_order'], fcfg['out_mode'])
    pred_out = delay_series(pred_out, fs_hz, spec['delay_s'])

    id_interp_nm = np.interp(t_aligned, t_id, id_moment_nm, left=np.nan, right=np.nan)
    applied_interp_nm = np.interp(t_aligned, t_raw.astype(np.float64), applied_nm.astype(np.float64), left=np.nan, right=np.nan)
    gt_nmpkg = (id_interp_nm - applied_interp_nm) / mass
    gt_nmpkg = lpf_nan(gt_nmpkg, fs_hz, GT_LPF_CUTOFF, GT_LPF_ORDER, GT_LPF_MODE)
    pred_out = lpf_nan(pred_out, fs_hz, GT_LPF_CUTOFF, GT_LPF_ORDER, GT_LPF_MODE)

    rmse, r2 = rmse_r2(gt_nmpkg, pred_out)
    return {
        'trial': stem,
        'exo_kind': exo_kind,
        'joint': spec['joint'],
        't': t_aligned.astype(np.float64),
        'gt_nmpkg': gt_nmpkg.astype(np.float64),
        'pred_nmpkg': pred_out.astype(np.float64),
        'rmse_nmpkg': rmse,
        'r2_nmpkg': r2,
        'offset_s': offset_s,
        'applied_key': applied_key,
        'gpio_key': gpio_key,
        'mass_kg': mass,
    }


TRIAL_DATA = {}
rows = []
for exo_kind, files in CANDIDATES.items():
    for fp in files:
        try:
            res = evaluate_exo_trial(fp, exo_kind)
            if res is None:
                continue
            TRIAL_DATA[res['trial']] = res
            rows.append({k: res[k] for k in ('trial', 'exo_kind', 'joint', 'rmse_nmpkg', 'r2_nmpkg', 'offset_s')})
            print(f"OK  {res['trial']} | RMSE={res['rmse_nmpkg']:.4f} R²={res['r2_nmpkg']:.4f} | offset={res['offset_s']:+.3f}s")
        except Exception as exc:
            WARNINGS.append(f'[WARN] {fp.name}: {exc}')
            print(f'FAIL {fp.name}: {exc}')

for w in WARNINGS:
    print(w)

summary_df = pd.DataFrame(rows)
print(f'\nLoaded {len(TRIAL_DATA)} exo trials')
if not summary_df.empty:
    display(summary_df)
    print('\nPer exo type:')
    display(summary_df.groupby('exo_kind')[['rmse_nmpkg', 'r2_nmpkg']].mean())
    print(f"\nOverall: RMSE={summary_df['rmse_nmpkg'].mean():.4f} N·m/kg | R²={summary_df['r2_nmpkg'].mean():.4f}")

FAIL _ab01_jinwoo_hip_0p8mps_lg_exo_on.npz: Unknown subject token: _ab01
FAIL _ab01_jinwoo_hip_0p8mps_ra_exo_on.npz: Unknown subject token: _ab01
FAIL _ab01_jinwoo_hip_1p2mps_lg_exo_on.npz: Unknown subject token: _ab01
FAIL _ab01_jinwoo_hip_1p6mps_lg_exo_on.npz: Unknown subject token: _ab01
FAIL _ab02_oscar_hip_0p8mps_lg_exo_on.npz: Unknown subject token: _ab02
FAIL _ab02_oscar_hip_0p8mps_ra_exo_on.npz: Unknown subject token: _ab02
FAIL _ab02_oscar_hip_1p2mps_lg_exo_on.npz: Unknown subject token: _ab02
FAIL _ab02_oscar_hip_1p6mps_lg_exo_on.npz: Unknown subject token: _ab02
OK  ab02_oscar_hip_0p8mps_lg_exo_on | RMSE=0.6807 R²=-1.0925 | offset=-0.188s


KeyboardInterrupt: 

In [ ]:
if not TRIAL_DATA:
    raise RuntimeError('No exo trials loaded. Check warnings above.')

trial_dd = widgets.Dropdown(options=sorted(TRIAL_DATA), description='Trial:')
time_slider = widgets.FloatRangeSlider(description='Time (s):', continuous_update=False, layout=widgets.Layout(width='700px'))
out = widgets.Output()


def _set_slider(trial_key):
    t = TRIAL_DATA[trial_key]['t']
    t_rel = t - np.nanmin(t)
    time_slider.min, time_slider.max = float(t_rel[0]), float(t_rel[-1])
    time_slider.step = max((time_slider.max - time_slider.min) / 500, 1e-3)
    time_slider.value = (time_slider.min, time_slider.max)


def _draw(trial_key, t_window):
    d = TRIAL_DATA[trial_key]
    t = d['t'] - np.nanmin(d['t'])
    t0, t1 = t_window
    m = (t >= t0) & (t <= t1)
    gt, pred = d['gt_nmpkg'], d['pred_nmpkg']

    fig, axs = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axs[0].plot(t[m], gt[m], color='#1e88e5', lw=2.0, label=f"GT (ID − {d['applied_key']})")
    axs[0].plot(t[m], pred[m], color='#e53935', lw=1.6, ls='--', label='Model')
    axs[0].set_ylabel('N·m/kg')
    axs[0].set_title(
        f"{trial_key} ({d['joint']}) | RMSE={d['rmse_nmpkg']:.4f} R²={d['r2_nmpkg']:.4f} | "
        f"GPIO rising-edge offset={d['offset_s']:+.3f}s"
    )
    axs[0].legend()
    axs[0].axhline(0, color='gray', lw=0.6, ls=':')

    axs[1].plot(t[m], (pred - gt)[m], color='#9c27b0', lw=1.4, label='Model − GT')
    axs[1].axhline(0, color='black', lw=0.8, ls=':')
    axs[1].set_ylabel('Residual (N·m/kg)')
    axs[1].set_xlabel('Time (s)')
    axs[1].legend()
    fig.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()


def _redraw(*_):
    _draw(trial_dd.value, time_slider.value)


def _on_trial(change):
    _set_slider(change['new'])
    _redraw()

trial_dd.observe(_on_trial, names='value')
time_slider.observe(_redraw, names='value')
_set_slider(trial_dd.value)
display(widgets.VBox([trial_dd, time_slider, out]))
_redraw()